In [1]:
import pandas as pd
import numpy as np
import hashlib

In [2]:
disaster_df = pd.read_csv("global_climate_events_economic_impact_2020_2025.csv")
gdp_df = pd.read_csv("World_GDP_Population_CO2_Emissions_Dataset.csv")

In [3]:
disaster_df.head()

,event_id,date,year,month,country,event_type,severity,duration_days,affected_population,deaths,injuries,economic_impact_million_usd,infrastructure_damage_score,response_time_hours,international_aid_million_usd,latitude,longitude,total_casualties,impact_per_capita,aid_percentage
0,EV01539,2020-01-01,2020,1,Japan,Tsunami,1,1,420956,0,2,0.01,4.9,11,0.0,85.4321,138.7206,2,0.02,0.0
1,EV02303,2020-01-01,2020,1,Qatar,Hurricane,1,4,3276,1,10,0.00,3.4,5,0.0,-32.0370,14.0111,11,0.00,0.0
2,EV01796,2020-01-02,2020,1,Canada,Drought,3,6,120382,0,9,0.10,8.9,10,0.0,78.4213,-112.7556,9,0.83,0.0
3,EV00175,2020-01-02,2020,1,Poland,Heatwave,6,16,185527,2,37,1.27,17.8,7,0.0,73.6564,115.0650,39,6.85,0.0
4,EV01115,2020-01-03,2020,1,UAE,Wildfire,4,16,176642,2,27,2.01,18.7,17,0.0,52.6458,101.5023,29,11.38,0.0


In [4]:
gdp_df.head()

,Year,GDP Real (USD),GDP growth (%),Per Capita,World Population,Net Change,Population change (%),Fossil CO2 Emissions (tons),CO2 emissions change,CO2 emissions per capita,Population Density (P/Km²)
0,2022,9.080000e+13,0.0324,11317,8021407192,66958801,0.0084,38521997860,0.0115,4.80,54
1,2021,8.790000e+13,0.0635,11054,7954448391,67447099,0.0086,38082163770,0.0595,4.79,53
2,2020,8.270000e+13,-0.0288,10483,7887001292,75707594,0.0097,35944470190,-0.0497,4.56,53
3,2019,8.510000e+13,0.0268,10898,7811293698,81390917,0.0105,37824905990,-0.0002,4.84,52
4,2018,8.290000e+13,0.0328,10726,7729902781,84284827,0.0110,37831867370,0.0255,4.89,52


In [5]:
def sha256_checksum(path):
    sha = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(4096), b""):
            sha.update(block)
    return sha.hexdigest()

disaster_checksum = sha256_checksum("global_climate_events_economic_impact_2020_2025.csv")
gdp_checksum = sha256_checksum("World_GDP_Population_CO2_Emissions_Dataset.csv")

In [6]:
print("SHA-256 - disaster data:", disaster_checksum)
print("SHA-256 - GDP/CO2 data:", gdp_checksum)


SHA-256 - disaster data: 27eb23bf938ea866120068dae9ccd3f0539bab39b02cd475f42511950f48328e
SHA-256 - GDP/CO2 data: 1d5b4ae7c57914b02df77a37fcf8201c1c9179d46c1c7af3cf4e17ed8e3527ca


In [7]:

print(disaster_df.info())
print("\nMissing values:")
print(disaster_df.isna().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 20 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   event_id                       3000 non-null   object 
 1   date                           3000 non-null   object 
 2   year                           3000 non-null   int64  
 3   month                          3000 non-null   int64  
 4   country                        3000 non-null   object 
 5   event_type                     3000 non-null   object 
 6   severity                       3000 non-null   int64  
 7   duration_days                  3000 non-null   int64  
 8   affected_population            3000 non-null   int64  
 9   deaths                         3000 non-null   int64  
 10  injuries                       3000 non-null   int64  
 11  economic_impact_million_usd    3000 non-null   float64
 12  infrastructure_damage_score    3000 non-null   f

In [8]:

print(gdp_df.info())
print("\nMissing values:")
print(gdp_df.isna().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46 entries, 0 to 45
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Year                         46 non-null     int64  
 1   GDP Real (USD)               46 non-null     float64
 2   GDP growth (%)               46 non-null     float64
 3   Per Capita                   46 non-null     int64  
 4   World Population             46 non-null     int64  
 5   Net Change                   46 non-null     int64  
 6   Population change (%)        46 non-null     float64
 7   Fossil CO2 Emissions (tons)  46 non-null     int64  
 8   CO2 emissions change         46 non-null     float64
 9   CO2 emissions per capita     46 non-null     float64
 10  Population Density (P/Km²)   46 non-null     int64  
dtypes: float64(5), int64(6)
memory usage: 4.1 KB
None

Missing values:
Year                           0
GDP Real (USD)                 0
GDP g

In [9]:
disaster_df["date"] = pd.to_datetime(disaster_df["date"], errors="coerce")


disaster_df["impact_per_person_usd"] = (
    disaster_df["economic_impact_million_usd"] * 1_000_000
    / disaster_df["affected_population"].replace(0, np.nan)
)

disaster_df["impact_per_person_usd"].replace([np.inf, -np.inf], np.nan, inplace=True)


num_cols_disaster = [
    "severity",
    "duration_days",
    "affected_population",
    "deaths",
    "injuries",
    "economic_impact_million_usd",
    "impact_per_person_usd",
]

display(disaster_df[num_cols_disaster].describe())


,severity,duration_days,affected_population,deaths,injuries,economic_impact_million_usd,impact_per_person_usd
count,3000.000000,3000.000000,3.000000e+03,3000.000000,3000.000000,3000.000000,3000.000000
mean,3.786333,8.783000,8.685505e+05,4.615000,39.228333,1.944367,3.613954
std,2.005165,14.714508,3.009690e+06,11.021491,78.812194,15.734990,10.779855
min,1.000000,0.000000,6.220000e+02,0.000000,0.000000,0.000000,0.000000
25%,2.000000,1.000000,5.451775e+04,1.000000,10.000000,0.010000,0.082870
50%,4.000000,2.000000,1.784770e+05,2.000000,18.000000,0.090000,0.483497
75%,5.000000,9.000000,6.082012e+05,3.000000,27.000000,0.530000,2.135447
max,9.000000,115.000000,5.624832e+07,117.000000,734.000000,718.210000,181.615708


In [10]:
gdp_df.columns = [c.strip() for c in gdp_df.columns]

gdp_df = gdp_df.rename(
    columns={
        "Year": "year",
        "GDP Real (USD)": "gdp_real_usd",
        "GDP growth (%)": "gdp_growth_pct",
        "Per Capita": "gdp_per_capita_usd",
        "World Population": "world_population",
        "Population change (%)": "population_change_pct",
        "Fossil CO2 Emissions (tons)": "co2_tons",
        "CO2 emissions change": "co2_change",
        "CO2 emissions per capita": "co2_per_capita",
        "Population Density (P/Km²)": "population_density_p_km2",
        "Net Change": "net_change"
    }
)


In [11]:
gdp_df["year"] = gdp_df["year"].astype(int)

In [32]:
gdp_df.head()

,year,gdp_real_usd,gdp_growth_pct,gdp_per_capita_usd,world_population,net_change,population_change_pct,co2_tons,co2_change,co2_per_capita,population_density_p_km2
0,2022,9.080000e+13,0.0324,11317,8021407192,66958801,0.0084,38521997860,0.0115,4.80,54
1,2021,8.790000e+13,0.0635,11054,7954448391,67447099,0.0086,38082163770,0.0595,4.79,53
2,2020,8.270000e+13,-0.0288,10483,7887001292,75707594,0.0097,35944470190,-0.0497,4.56,53
3,2019,8.510000e+13,0.0268,10898,7811293698,81390917,0.0105,37824905990,-0.0002,4.84,52
4,2018,8.290000e+13,0.0328,10726,7729902781,84284827,0.0110,37831867370,0.0255,4.89,52


In [34]:
num_cols_gdp = [
    "gdp_real_usd",
    "gdp_growth_pct",
    "gdp_per_capita_usd",
    "world_population",
    "population_change_pct",
    "co2_tons",
    "co2_change",
    "co2_per_capita",
    "population_density_p_km2",
]
gdp_df[num_cols_gdp].describe()

,gdp_real_usd,gdp_growth_pct,gdp_per_capita_usd,world_population,population_change_pct,co2_tons,co2_change,co2_per_capita,population_density_p_km2
count,4.600000e+01,46.000000,46.000000,4.600000e+01,46.000000,4.600000e+01,46.000000,46.000000,46.000000
mean,5.097609e+13,0.030350,8006.326087,6.128450e+09,0.014463,2.745007e+10,0.016337,4.440435,41.152174
std,2.013534e+13,0.015413,1720.809526,1.150810e+09,0.002909,6.724902e+09,0.020672,0.312644,7.720010
min,2.400000e+13,-0.028800,5685.000000,4.217864e+09,0.008400,1.900807e+10,-0.049700,4.030000,28.000000
25%,3.412500e+13,0.026425,6612.000000,5.165102e+09,0.012725,2.204355e+10,0.006125,4.150000,35.000000
50%,4.735000e+13,0.032350,7729.000000,6.130355e+09,0.013600,2.522144e+10,0.016550,4.365000,41.000000
75%,6.665000e+13,0.039950,9398.750000,7.088626e+09,0.017800,3.464557e+10,0.031150,4.780000,47.750000
max,9.080000e+13,0.063500,11317.000000,8.021407e+09,0.018500,3.852200e+10,0.061200,4.950000,54.000000


In [35]:
yearly_disasters = (
    disaster_df
    .groupby("year")
    .agg(
        n_events=("event_id", "count"),
        total_econ_impact_musd=("economic_impact_million_usd", "sum"),
        avg_severity=("severity", "mean"),
        total_deaths=("deaths", "sum"),
        total_injuries=("injuries", "sum"),
        total_affected=("affected_population", "sum"),
    )
    .reset_index()
)

In [36]:
display(yearly_disasters)


,year,n_events,total_econ_impact_musd,avg_severity,total_deaths,total_injuries,total_affected
0,2020,523,1972.30,3.820268,2609,22372,378672205
1,2021,508,1005.49,3.732283,2188,19946,440510810
2,2022,500,879.02,3.894000,2576,21315,469955649
3,2023,553,590.17,3.714286,2230,18571,403633950
4,2024,542,850.61,3.850554,2478,20581,577833392
5,2025,374,535.51,3.681818,1764,14900,335045480


In [37]:
climate_econ_yearly = yearly_disasters.merge(gdp_df, on="year", how="inner")

In [38]:
display(climate_econ_yearly)

,year,n_events,total_econ_impact_musd,avg_severity,total_deaths,total_injuries,total_affected,gdp_real_usd,gdp_growth_pct,gdp_per_capita_usd,world_population,net_change,population_change_pct,co2_tons,co2_change,co2_per_capita,population_density_p_km2
0,2020,523,1972.30,3.820268,2609,22372,378672205,8.270000e+13,-0.0288,10483,7887001292,75707594,0.0097,35944470190,-0.0497,4.56,53
1,2021,508,1005.49,3.732283,2188,19946,440510810,8.790000e+13,0.0635,11054,7954448391,67447099,0.0086,38082163770,0.0595,4.79,53
2,2022,500,879.02,3.894000,2576,21315,469955649,9.080000e+13,0.0324,11317,8021407192,66958801,0.0084,38521997860,0.0115,4.80,54


In [40]:
!pip install duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 22.9 MB/s  0:00:00m0:00:0100:01


In [42]:
import duckdb

con = duckdb.connect("climate_economy.duckdb")

In [44]:
con.register("disasters_df", disaster_df)
con.register("gdp_df", gdp_df)
con.register("climate_yearly_df", climate_econ_yearly)

con.execute("CREATE OR REPLACE TABLE disasters AS SELECT * FROM disasters_df;")
con.execute("CREATE OR REPLACE TABLE world_gdp AS SELECT * FROM gdp_df;")
con.execute("CREATE OR REPLACE TABLE climate_yearly AS SELECT * FROM climate_yearly_df;")

In [45]:
con.execute("SHOW TABLES;").df()

,name
0,climate_yearly
1,climate_yearly_df
2,disasters
3,disasters_df
4,gdp_df
5,world_gdp


In [46]:
con.execute("""
    SELECT year, COUNT(*) AS n_events
    FROM disasters
    GROUP BY year
    ORDER BY year;
""").df()

,year,n_events
0,2020,523
1,2021,508
2,2022,500
3,2023,553
4,2024,542
5,2025,374


In [47]:
con.execute("""
    SELECT year, COUNT(*) AS n_events
    FROM disasters
    GROUP BY year
    ORDER BY year;
""").df()

,year,n_events
0,2020,523
1,2021,508
2,2022,500
3,2023,553
4,2024,542
5,2025,374


In [48]:
con.execute("""
    SELECT year, COUNT(*) AS n_events
    FROM disasters
    GROUP BY year
    ORDER BY year;
""").df()

,year,n_events
0,2020,523
1,2021,508
2,2022,500
3,2023,553
4,2024,542
5,2025,374


In [49]:
integrated_query = con.execute("""
    SELECT 
        d.year,
        SUM(d.economic_impact_million_usd) AS total_loss_musd,
        SUM(d.deaths) AS total_deaths,
        SUM(d.injuries) AS total_injuries,
        AVG(d.severity) AS avg_severity,
        g.gdp_real_usd,
        g.world_population,
        g.co2_tons,
        g.co2_per_capita
    FROM disasters d
    LEFT JOIN world_gdp g
        ON d.year = g.year
    GROUP BY d.year, g.gdp_real_usd, g.world_population, g.co2_tons, g.co2_per_capita
    ORDER BY d.year;
""").df()

In [50]:
integrated_query.head()

,year,total_loss_musd,total_deaths,total_injuries,avg_severity,gdp_real_usd,world_population,co2_tons,co2_per_capita
0,2020,1972.30,2609.0,22372.0,3.820268,8.270000e+13,7887001292,35944470190,4.56
1,2021,1005.49,2188.0,19946.0,3.732283,8.790000e+13,7954448391,38082163770,4.79
2,2022,879.02,2576.0,21315.0,3.894000,9.080000e+13,8021407192,38521997860,4.80
3,2023,590.17,2230.0,18571.0,3.714286,NaN,<NA>,<NA>,NaN
4,2024,850.61,2478.0,20581.0,3.850554,NaN,<NA>,<NA>,NaN


In [51]:
con.register("integrated_df", integrated_query)
con.execute("CREATE OR REPLACE TABLE integrated_yearly AS SELECT * FROM integrated_df;")

In [52]:
con.execute("SHOW TABLES;").df()

,name
0,climate_yearly
1,climate_yearly_df
2,disasters
3,disasters_df
4,gdp_df
5,integrated_df
6,integrated_yearly
7,world_gdp


In [53]:
con.close()